# 《实用Python编程》教学代码
## 第5章 数据智能 Sklearn机器学习

## 案例1: 英文影评情感识别

### 示例代码5.1 使用Pandas读取IMDB影评情感识别数据

In [1]:
import pandas as pd
import csv

data_file = 'pybook-data/ch5/imdb_labelled.txt' #指定数据文件路径
df = pd.read_csv(data_file, names=["sentence", "label"], sep="\t", quoting=csv.QUOTE_NONE)  
print(df.shape) # 输出 (1000, 2)
sents = df['sentence'].values #读取影评
y = df['label'].values #读取每条影评对应的二值标签

(1000, 2)


In [6]:
df

,sentence,label
0,"A very, very, very slow-moving, aimless movie ...",0
1,Not sure who was more lost - the flat characte...,0
2,Attempting artiness with black & white and cle...,0
3,Very little music or anything to speak of.,0
4,The best scene in the movie was when Gerardo i...,1
...,...,...
995,I just got bored watching Jessice Lange take h...,0
996,"Unfortunately, any virtue in this film's produ...",0
997,"In a word, it is embarrassing.",0
998,Exceptionally bad!,0


### 示例代码5.2 使用train_test_split划分影评情感识别数据

In [7]:
import os
from sklearn.model_selection import train_test_split

out_dir = 'output'
train_file = f'{out_dir}/imdb_labelled_train.csv'
test_file = f'{out_dir}/imdb_labelled_test.csv'
if os.path.exists(train_file) and os.path.exists(test_file): #若文件已存在，直接导入
    print(f'Files exist at "{train_file}" and "{test_file}"')
    df_train, df_test = pd.read_csv(train_file), pd.read_csv(test_file)
    sents_train, sents_test = df_train['sentence'].values, df_test['sentence'].values
    y_train, y_test = df_train['label'].values, df_test['label'].values
else:
    os.makedirs(out_dir, exist_ok=True)
    sents_train, sents_test, y_train, y_test = train_test_split(sents,y,test_size=0.2,random_state=1)
    print(f'训练样本数量：{len(sents_train)}')  # 800
    print(f'测试样本数量：{len(sents_test)}')  # 200
    # 将划分后的数据保存到文件，以便重复使用该划分方案
    df_train = pd.DataFrame({'sentence': sents_train, 'label': y_train})
    df_test = pd.DataFrame({'sentence': sents_test, 'label': y_test})
    # 保存为CSV文件
    df_train.to_csv(train_file, index=False)
    df_test.to_csv(test_file, index=False)

训练样本数量：800
测试样本数量：200


### 示例代码5.3 提取影评文本的BoW特征

In [8]:
from sklearn.feature_extraction.text import CountVectorizer
# 使用训练数据构建BoW特征提取器
fe = CountVectorizer()
fe.fit(sents_train)
vob = fe.get_feature_names_out()
print(f'词表大小：{len(vob)}') # 词表大小：2647
print(f'部分词汇：{vob[-5:]}') # 部分词汇：['youthful' 'youtube' 'yun' 'zillion' 'zombiez']
# 根据词表提取BoW特征
x_train = fe.transform(sents_train)
x_test = fe.transform(sents_test)
print(f'训练集特征规模：{x_train.shape}') # 训练集特征规模：(800, 2647) 
print(f'测试集特征规模：{x_test.shape}')  # 测试集特征规模：(200, 2647)

词表大小：2647
部分词汇：['youthful' 'youtube' 'yun' 'zillion' 'zombiez']
训练集特征规模：(800, 2647)
测试集特征规模：(200, 2647)


### 示例代码5.4 查看BoW特征

In [9]:
i = 2
print(df_train['sentence'][i]) # It's a sad movie, but very good.  
values = x_train[i].data
columns = x_train[i].indices
print(vob[columns], values) 

It's a sad movie, but very good.  
['but' 'good' 'it' 'movie' 'sad' 'very'] [1 1 1 1 1 1]


### 示例代码5.5 基于sklearn训练用于英文影评情感识别的LRC并保存

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
import joblib 

clf = LogisticRegression() # 构建模型
clf.fit(x_train, y_train) # 训练模型
full_model = make_pipeline(fe, clf) # 封装特征提取器与分类器
joblib.dump(full_model, 'movie_review_bow_lrc.joblib') # 保存为一个本地文件

['movie_review_bow_lrc.joblib']

### 示例代码5.6 使用LRC在已提取BoW特征的影评情感测试集上预测

In [14]:
y_pred = clf.predict(x_test)
probs = clf.predict_proba(x_test)
print(y_pred.shape, probs.shape)

(200,) (200, 2)


### 示例代码5.7 使用DataFrame表格观察LRC分类结果

In [16]:
import numpy as np

df_test['negative'] = probs[:,0]
df_test['positive'] = probs[:,1]
df_test['argmax'] = np.argmax(probs, axis=1)
df_test['y_pred'] = y_pred
df_test

,sentence,label,negative,positive,argmax,y_pred
0,"By the time the film ended, I not only dislike...",0,0.823776,0.176224,0,0
1,"The acting, as you'd expect from this cast, is...",1,0.536713,0.463287,0,0
2,"The plot doesn't hang together at all, and the...",0,0.746511,0.253489,0,0
3,I felt as though her going to Ireland did abso...,0,0.454886,0.545114,1,1
4,"If good intentions made a film great, then thi...",1,0.037038,0.962962,1,1
...,...,...,...,...,...,...
195,I hate writing bad reviews about films - espec...,0,0.998442,0.001558,0,0
196,What the hell kind of crap is that?!,0,0.727868,0.272132,0,0
197,I don't think there are any redeeming qualitie...,0,0.781497,0.218503,0,0
198,And the accents are absolutely abysmal!,0,0.404990,0.595010,1,1


### 示例代码5.8 加载流水线模型并对文本输入进行预测

In [17]:
full_model = joblib.load('movie_review_bow_lrc.joblib')

input_texts = ['this is a good movie', 'the movie is horrible', 'a horrible movie', 'not a bad movie']
prob = full_model.predict_proba(input_texts)
pred = np.argmax(prob, axis=1)
names = ['负面评论', '正面评论']

for i, text in enumerate(input_texts):
    j = pred[i]
    print(f'{text} -> {names[j]} ({prob[i][j]:.2f})')

this is a good movie -> 正面评论 (0.66)
the movie is horrible -> 负面评论 (0.68)
a horrible movie -> 负面评论 (0.68)
not a bad movie -> 负面评论 (0.97)


### 示例代码5.9 通过DataFrame关联单词与对应的LRC系数

In [23]:
vob = fe.get_feature_names_out()
weights = clf.coef_.ravel()
word_weight_table = pd.DataFrame({'单词':vob, 'LRC系数': weights})
sorted_table = word_weight_table.sort_values(by='LRC系数', ascending=False)

In [21]:
sorted_table[:10]

,单词,LRC系数
2602,wonderful,1.420662
1015,great,1.334520
213,beautiful,1.284007
1395,love,1.252346
1353,liked,1.246554
952,funny,1.233151
743,enjoyed,1.203274
1910,right,1.098224
51,actually,0.973320
1212,interesting,0.971396


In [22]:
sorted_table[-10:]

,单词,LRC系数
1569,not,-0.951759
610,didn,-0.960953
383,cheap,-1.017008
2212,sucked,-1.049227
1413,make,-1.065150
2201,stupid,-1.145529
2621,would,-1.206083
1559,no,-1.345398
182,awful,-1.390197
189,bad,-2.309000


### 示例代码5.10 计算分类准确率、混淆矩阵，给出分类性能报告

In [24]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

accuracy  = accuracy_score(y_test, y_pred)
print (f'分类准确率：{accuracy:.3f}')
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
report = classification_report(y_test, y_pred, digits=3)
print(report)

分类准确率：0.815
              precision    recall  f1-score   support

           0      0.840     0.800     0.820       105
           1      0.790     0.832     0.810        95

    accuracy                          0.815       200
   macro avg      0.815     0.816     0.815       200
weighted avg      0.816     0.815     0.815       200



## 案例2: 中文酒店评论情感识别

### 示例代码5.11 使用pandas读取中文酒店评论情感识别数据

In [30]:
import pandas as pd

data_file = 'pybook-data/ch5/ChnSentiCorp_htl_all.csv'
df = pd.read_csv(data_file)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
df

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7760,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7761,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7762,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7763,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


### 示例代码5.12 使用train_test_split划分酒店评论情感识别数据集

In [31]:
out_dir = 'output'
train_file = f'{out_dir}/hotel_labelled_train.csv'
test_file = f'{out_dir}/hotel_labelled_test.csv'
if os.path.exists(train_file) and os.path.exists(test_file): #若文件已存在，直接导入
    print(f'Files exist at "{train_file}" and "{test_file}"')
    df_train, df_test  = pd.read_csv(train_file), pd.read_csv(test_file)
    sents_train, sents_test = df_train['review'].values, df_test['review'].values
    y_train, y_test = df_train['label'].values, df_test['label'].values
else:
    os.makedirs(out_dir, exist_ok=True)
    sents = df['review'].values
    y = df['label'].values
    sents_train, sents_test, y_train, y_test = train_test_split(sents,y,test_size=0.2,random_state=1)
    # 将划分后的数据保存到文件，以便重复使用该划分方案
    df_train = pd.DataFrame({'label': y_train, 'review': sents_train})
    df_test = pd.DataFrame({'label': y_test, 'review': sents_test})
    # 保存为CSV文件
    df_train.to_csv(train_file, index=False)
    df_test.to_csv(test_file, index=False)
print(f'训练样本数量：{len(sents_train)}') # 训练样本数量：6212
print(f'测试样本数量：{len(sents_test)}')  # 测试样本数量：1553

训练样本数量：6212
测试样本数量：1553


In [32]:
df_test

,label,review
0,1,酒店的环境还是挺不错的，一大片的高尔夫球场，绿色的草地让人心情很爽！旁边就是阳澄湖。房间还可...
1,0,房间比较脏，尤其是地毯，像80年代的东西。最令人难以忍受的是早餐，由于有旅行团，人比较多，而...
2,1,"第一次去香港,住帝豪奥运.之前也看了大家的评论,还是没有实际感受来得真切呵.我和老婆的一致评..."
3,1,我并没有实际前往，而是帮我经理预定的。但是我经理回来对这个酒店评价还可以，和我说NOBAD，...
4,1,"这个宾馆服务很不怎么样?当然,如果谈到房间内部肯定非常好,比HYYYAT,喜来登...,都要..."
...,...,...
1548,1,酒店整体还算不错，就是早餐券的购买方式太不合理，又贵，买了又不能退，最后被逼到酒店后的永和豆...
1549,1,服务一流，门口公交四通八达，早餐丰富，值得一住。我到厦门五天，两天锦江之星，两天厦门宾馆，都...
1550,0,这个酒店的设施虽然简单，但服务态度恶劣，一点都不尊重客人！我是5月21日通过携程网预定的房间...
1551,1,住过两次了，但真的没给人留下什么印象。早餐还比较丰盛，整体服务也还不错。湖南的酒店服务业感觉...


### 示例代码5.13 提取中文文本的BoW特征

In [34]:
import jieba

jieba.initialize()
def jieba_tokenize(text):
    return jieba.lcut(text)

fe = CountVectorizer(tokenizer=jieba_tokenize)
fe.fit(sents_train)
vob = fe.get_feature_names_out()
print('词表大小:', len(vob)) # 词表大小: 26262
print('部分特征:', vob[10000:10005]) # 部分特征: ['安贞桥' '安逸' '安阳' '安静' '安静下来']

x_train = fe.transform(sents_train)
x_test = fe.transform(sents_test)

print(f'训练集特征规模：{x_train.shape}') # 训练集特征规模：(6212, 26262) 
print(f'测试集特征规模：{x_test.shape}') # 测试集特征规模：(1553, 26262)

词表大小: 26262
部分特征: ['安贞桥' '安逸' '安阳' '安静' '安静下来']
训练集特征规模：(6212, 26262)
测试集特征规模：(1553, 26262)


### 示例代码5.14 训练用于中文酒店评论情感识别的LRC并保存

In [35]:
clf = LogisticRegression(max_iter=1000) 
clf.fit(x_train, y_train)
# 以下为可选操作
full_model = make_pipeline(fe, clf)
joblib.dump(full_model, 'hotel_review_bow_lrc.joblib') 

['hotel_review_bow_lrc.joblib']

### 示例代码5.15 酒店评论情感识别与评估

In [36]:
y_pred = clf.predict(x_test)
accuracy  = accuracy_score(y_test, y_pred)
print (f'分类准确率：{accuracy:.3f}')
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
report = classification_report(y_test, y_pred, digits=3)
print(report)

分类准确率：0.875
              precision    recall  f1-score   support

           0      0.807     0.779     0.793       476
           1      0.904     0.917     0.911      1077

    accuracy                          0.875      1553
   macro avg      0.855     0.848     0.852      1553
weighted avg      0.874     0.875     0.874      1553



### 加载流水线模型并对文本输入进行预测

In [40]:
full_model = joblib.load('hotel_review_bow_lrc.joblib')

input_texts = ['房间干净整洁，服务态度超好。', 
               '位置很方便，早餐也很丰盛。', 
               '隔音很差，一晚上没睡好', 
               '房间有异味，前台态度冷漠。']

prob = full_model.predict_proba(input_texts)
pred = np.argmax(prob, axis=1)
names = ['负面评论', '正面评论']

for i, text in enumerate(input_texts):
    j = pred[i]
    print(f'{text} -> {names[j]} ({prob[i][1]:.2f})')

房间干净整洁，服务态度超好。 -> 正面评论 (0.79)
位置很方便，早餐也很丰盛。 -> 正面评论 (0.93)
隔音很差，一晚上没睡好 -> 负面评论 (0.26)
房间有异味，前台态度冷漠。 -> 负面评论 (0.34)
